# 8-seed attention-LSTM training on full data (Keras 3 compatible).

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import time

BASE = '..'

class AttentionLayer(layers.Layer):
    def __init__(self, units=20, return_sequences=True, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.return_sequences = return_sequences

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], self.units),
            initializer='glorot_uniform', trainable=True, name='attention_weight')
        self.b = self.add_weight(shape=(self.units,),
            initializer='zeros', trainable=True, name='attention_bias')
        self.u = self.add_weight(shape=(self.units,),
            initializer='glorot_uniform', trainable=True, name='attention_vector')
        super().build(input_shape)

    def call(self, inputs):
        score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        attention_weights = tf.nn.softmax(tf.tensordot(score, self.u, axes=1), axis=1)
        attention_weights = tf.expand_dims(attention_weights, axis=-1)
        context_vector = tf.reduce_sum(inputs * attention_weights, axis=1)
        if self.return_sequences:
            return context_vector, tf.squeeze(attention_weights, axis=-1)
        else:
            return context_vector

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'return_sequences': self.return_sequences})
        return config

def build_model(input_shape, lstm_units=20, attention_units=20):
    inputs = Input(shape=input_shape)
    lstm_out = layers.LSTM(lstm_units, return_sequences=True)(inputs)
    ctx, attn = AttentionLayer(units=attention_units, return_sequences=True)(lstm_out)
    outputs = layers.Dense(1)(ctx)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Load data
print("Loading data...")
X_train = np.load(f'{BASE}/results/X_train.npy')
y_train = np.load(f'{BASE}/results/y_train.npy')
X_test = np.load(f'{BASE}/results/X_test.npy')
y_test = np.load(f'{BASE}/results/y_test.npy')
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

SEEDS = [42, 123, 456, 789, 1011, 1213, 1415, 1617]
results = []

for i, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"Seed {seed} ({i+1}/{len(SEEDS)})")
    print(f"{'='*60}")
    t0 = time.time()

    np.random.seed(seed)
    tf.random.set_seed(seed)

    model = build_model((X_train.shape[1], X_train.shape[2]))

    val_size = int(len(X_train) * 0.1)
    X_val = X_train[-val_size:]
    y_val = y_train[-val_size:]
    X_tr = X_train[:-val_size]
    y_tr = y_train[:-val_size]

    callbacks = [EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)]

    history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                       epochs=50, batch_size=80, callbacks=callbacks, verbose=0)

    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    elapsed = time.time() - t0

    result = {'seed': seed, 'mae': mae, 'rmse': rmse, 'r2': r2,
              'epochs': len(history.history['loss']), 'time_sec': elapsed}
    results.append(result)
    print(f"  MAE={mae:.6f}, RMSE={rmse:.6f}, R²={r2:.6f}, epochs={result['epochs']}, time={elapsed:.0f}s")

    del model
    tf.keras.backend.clear_session()

# Save results
df = pd.DataFrame(results)
output_path = f'{BASE}/results/tables/attention_8seed_fulldata.csv'
df.to_csv(output_path, index=False)
print(f"\n{'='*60}")
print(f"Results saved to {output_path}")
print(f"{'='*60}")
print(df.to_string(index=False))
print(f"\nSummary:")
print(f"  Converged (R²>0): {(df['r2']>0).sum()}/{len(df)} ({(df['r2']>0).mean()*100:.1f}%)")
print(f"  R²>0.5: {(df['r2']>0.5).sum()}/{len(df)} ({(df['r2']>0.5).mean()*100:.1f}%)")
print(f"  R²>0.7: {(df['r2']>0.7).sum()}/{len(df)} ({(df['r2']>0.7).mean()*100:.1f}%)")
print(f"  Mean MAE: {df['mae'].mean():.6f} ± {df['mae'].std():.6f}")
print(f"  Mean R²: {df['r2'].mean():.6f} ± {df['r2'].std():.6f}")
print(f"  Best R²: {df['r2'].max():.6f} (seed {df.loc[df['r2'].idxmax(), 'seed']})")
print("DONE")